<div style="font-family:system-ui;border-left:6px solid #B8721A;padding:6px 16px">
<div style="font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#566273">The AI-Ready Organisation · Lab 2 · Module 2 · ≈40 min</div>
<div style="font-size:26px;font-weight:800;margin:6px 0">Custom model versus off-the-shelf forecasting</div>
<div style="font-size:15px"><b>You will:</b> compare two forecasts you could buy against one you would build, measure the accuracy gap, and turn it into a build, buy or blend recommendation.</div>
<div style="font-size:13px;color:#566273;margin-top:6px">Dataset: Walmart weekly sales for 45 stores, 2010–2012 (public Kaggle dataset)</div>
</div>

**How to use this notebook** · No coding needed.
1. In the menu choose **Runtime → Run all** (in Kaggle: **Run All**). The first run takes 1–3 minutes.
2. Change values in the **form fields** on the right of each cell (sliders, dropdowns). The cell reruns when you change a value; if it does not, press the play button.
3. Read the plain-English findings under each chart and record your answers in the final cell.

If the real dataset cannot be downloaded (for example, without a Kaggle sign-in), the notebook automatically switches to a **realistic simulated copy with the same structure** and tells you so. The lessons still hold.

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import warnings, os, io, zipfile, urllib.request
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
plt.rcParams.update({"figure.figsize": (9, 4.4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 11, "axes.titleweight": "bold"})
BUILD, BUY, BLEND, INK, GOOD, BAD = "#B8721A", "#13807B", "#4255B0", "#111822", "#2D7A4C", "#B03A3A"
pd.set_option("display.max_columns", 30)

def say(text):
    display(Markdown(text))

def scorecard(title, rows):
    """rows: list of (label, value, note)"""
    body = "".join(f"<tr><td style='padding:6px 12px'>{a}</td><td style='padding:6px 12px;font-weight:700;text-align:right'>{b}</td>"
                   f"<td style='padding:6px 12px;color:#566273'>{c}</td></tr>" for a, b, c in rows)
    display(HTML(f"<div style='font-family:system-ui;margin:8px 0'><div style='font-weight:700;font-size:15px;margin-bottom:4px'>{title}</div>"
                 f"<table style='border-collapse:collapse;font-size:14px'>{body}</table></div>"))

def try_kagglehub(handle, filename):
    """Download a public Kaggle dataset. Returns a file path or None."""
    try:
        try:
            import kagglehub
        except ImportError:
            os.system("pip -q install kagglehub")
            import kagglehub
        path = kagglehub.dataset_download(handle)
        for root, _, files in os.walk(path):
            if filename in files:
                return os.path.join(root, filename)
    except Exception as e:
        print(f"Kaggle download unavailable ({type(e).__name__}).")
    return None

RNG = np.random.default_rng(42)
DATA_NOTE = ""

## 1 · Load weekly store sales

In [ ]:
#@title Load data { display-mode: "form" }
p = try_kagglehub("yasserh/walmart-dataset", "Walmart.csv")
if p:
    df = pd.read_csv(p)
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
    DATA_NOTE = "Real Walmart weekly sales data."
else:
    dates = pd.date_range("2010-02-05", periods=143, freq="7D")
    rows = []
    for s in range(1, 46):
        base = RNG.uniform(3e5, 2.2e6); trend = RNG.normal(0, .0025)
        shock_at = RNG.integers(40, 120); shock = RNG.choice([1, 1, 1, .85, 1.18])
        for i, d in enumerate(dates):
            w = d.isocalendar().week
            season = 1 + .08*np.sin(2*np.pi*w/52) + (.35 if w in (47, 51) else 0) + (.12 if w in (48, 50) else 0)
            hol = int(w in (6, 36, 47, 52))
            lvl = base*(1+trend*i)*(shock if i >= shock_at else 1)
            rows.append((s, d, lvl*season*RNG.normal(1, .04), hol, RNG.normal(60, 18),
                         3 + .01*i + RNG.normal(0, .05), 170 + .1*i, 8 + RNG.normal(0, .5)))
    df = pd.DataFrame(rows, columns=["Store", "Date", "Weekly_Sales", "Holiday_Flag", "Temperature", "Fuel_Price", "CPI", "Unemployment"])
    DATA_NOTE = "Simulated copy (Kaggle download unavailable). Same columns as the Walmart dataset."
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)
df["week"] = df["Date"].dt.isocalendar().week.astype(int); df["year"] = df["Date"].dt.year
say(f"**{DATA_NOTE}** {df.Store.nunique()} stores × {df.Date.nunique()} weeks; total sales **${df.Weekly_Sales.sum()/1e9:.1f}bn**.")
display(df.head())

## 2 · Three ways to forecast the last 26 weeks
- **Option A · Buy (basic tool):** seasonal naive. Next year's week = same week last year. Many planning tools default to this.
- **Option B · Buy (AutoML default):** a gradient-boosting model on the raw columns with default settings, as a generic AutoML product would.
- **Option C · Build (custom):** the same algorithm, plus features a retailer's own team would engineer: last year's sales for that store and week, each store's seasonal profile, and pre-holiday build-up weeks.

All three forecast 26 weeks ahead using only information available at forecast time.

In [ ]:
#@title Run the three forecasts { display-mode: "form" }
from sklearn.ensemble import HistGradientBoostingRegressor
cut = df.Date.max() - pd.Timedelta(weeks=26)
g = df.groupby("Store")["Weekly_Sales"]
df["lag52"] = g.shift(52)
recent = g.transform(lambda s: s.shift(26).rolling(8).mean()); year_ago = g.transform(lambda s: s.shift(78).rolling(8).mean())
df["yoy_ratio"] = (recent / year_ago).clip(.6, 1.6)
df["trend_adj_lag52"] = df["lag52"] * df["yoy_ratio"]
train_mask = df.Date <= cut
prof = df[train_mask].groupby(["Store", "week"])["Weekly_Sales"].mean().rename("store_week_avg")
lvl = df[train_mask].groupby("Store")["Weekly_Sales"].mean().rename("store_level")
df = df.drop(columns=[c for c in ["store_week_avg", "store_level"] if c in df]).join(prof, on=["Store", "week"]).join(lvl, on="Store")
df["pre_holiday"] = df.week.isin([46, 47, 48, 50, 51]).astype(int)
tr, te = df[train_mask & df.lag52.notna()], df[~train_mask].copy()
te["A"] = te["lag52"]
raw = ["Store", "Holiday_Flag", "Temperature", "Fuel_Price", "CPI", "Unemployment", "week"]
mB = HistGradientBoostingRegressor(random_state=0, categorical_features=[0]).fit(tr[raw], tr.Weekly_Sales); te["B"] = mB.predict(te[raw])
eng = raw + ["lag52", "trend_adj_lag52", "yoy_ratio", "store_week_avg", "store_level", "pre_holiday"]
# the custom model predicts the ratio to trend-adjusted last year, a common retail planning technique
trc = tr.dropna(subset=["trend_adj_lag52"])
mC = HistGradientBoostingRegressor(random_state=0, categorical_features=[0], max_iter=300, learning_rate=.05).fit(trc[eng], trc.Weekly_Sales / trc.trend_adj_lag52)
te["C"] = mC.predict(te[eng].fillna({"trend_adj_lag52": te["lag52"], "yoy_ratio": 1})) * te["trend_adj_lag52"].fillna(te["lag52"])
wmape = lambda d, c: (d.Weekly_Sales - d[c]).abs().sum() / d.Weekly_Sales.sum() * 100
res = {c: wmape(te.dropna(subset=["A"]), c) for c in "ABC"}
scorecard("Forecast error (WMAPE, lower is better)", [
    ("A · Buy: seasonal naive", f"{res['A']:.1f}%", "no modelling"),
    ("B · Buy: AutoML default", f"{res['B']:.1f}%", "generic features"),
    ("C · Build: custom features", f"{res['C']:.1f}%", "retailer's own know-how")])
best_buy = min(res["A"], res["B"]); gap = best_buy - res["C"]
say(f"The custom build is **{gap:.1f} percentage points** more accurate than the best option you could buy." if gap > 0 else
    f"The custom build is **not** more accurate than the best option you could buy (gap {gap:.1f} points). That is itself a strong signal to buy.")

## 3 · Look at one store

In [ ]:
#@title Choose a store { display-mode: "form" }
store_id = 20 #@param {type:"slider", min:1, max:45, step:1}
s = df[df.Store == store_id]; t = te[te.Store == store_id]
fig, ax = plt.subplots(figsize=(10, 4.4))
ax.plot(s.Date, s.Weekly_Sales/1e3, color="#9AA5B1", lw=1.2, label="Actual")
ax.plot(t.Date, t.A/1e3, color="#9AA5B1", ls=":", lw=2, label="A · seasonal naive")
ax.plot(t.Date, t.B/1e3, color=BUY, lw=2, label="B · AutoML default")
ax.plot(t.Date, t.C/1e3, color=BUILD, lw=2.4, label="C · custom build")
ax.axvline(cut, color=INK, ls="--", lw=1); ax.set(title=f"Store {store_id}: weekly sales ($k)", ylabel="$k"); ax.legend(ncol=2, fontsize=9); plt.show()
per = te.groupby("Store").apply(lambda d: pd.Series({c: wmape(d, c) for c in "BC"}))
say(f"Across stores, the custom build beats AutoML in **{(per.C < per.B).mean():.0%}** of stores.")

## 4 · Is the gap worth building for?
Enter your own business assumptions. A rule of thumb in retail planning is that each point of forecast error carries inventory and markdown cost; use your finance team's figure if you have one.

In [ ]:
#@title Translate accuracy into money { display-mode: "form" }
annual_sales_m = 2500 #@param {type:"number"}
cost_per_error_point_pct = 0.08 #@param {type:"number"}
build_cost_5yr_k = 1350 #@param {type:"number"}
buy_cost_5yr_k = 1560 #@param {type:"number"}
value_5yr = max(gap, 0) * cost_per_error_point_pct/100 * annual_sales_m*1e6 * 5
extra = (build_cost_5yr_k - buy_cost_5yr_k) * 1e3
rating = 1 if gap < 1 else 2 if gap < 2 else 3 if gap < 4 else 4 if gap < 7 else 5
scorecard("Five-year view", [
    ("Value of the accuracy gap", f"${value_5yr/1e6:,.1f}m", f"{gap:.1f} pts × {cost_per_error_point_pct}% of sales per point"),
    ("Extra cost of building", f"${extra/1e6:,.1f}m", "negative means building is cheaper"),
    ("Net case for building", f"${(value_5yr - extra)/1e6:,.1f}m", ""),
    ("Suggested 'differentiation' rating", f"{rating} / 5", "enter this in Simulator 2.1")])
verdict = "Build" if (value_5yr - extra) > 0 and rating >= 4 else "Blend" if (value_5yr - extra) > 0 else "Buy"
say(f"### Indicative recommendation: **{verdict}**\nBuild = the gap is large and pays for the team. Blend = buy the forecasting platform and add your own features to it. Buy = the gap does not justify the effort.")

## Your reflection → capstone §2
Answer in the form fields, then run the cell. Copy the summary into §2 of your business case on the course page.

In [ ]:
#@title Record your answers { display-mode: "form" }
q1 = "" #@param {type:"string"}
q2 = "" #@param {type:"string"}
q3 = "" #@param {type:"string"}
answers = [("My build, buy or blend recommendation in two lines:", q1), ("Which custom feature drove the gap, and do competitors have the same data?", q2), ("What would change my mind?", q3)]
say("### Lab 2 reflection")
for q, a in answers:
    say(f"**{q}**  \n{a if a.strip() else '_(not answered yet)_'}")